# Polynomial Regression - Quick Revision Notes

Use Polynomial Regression when the data follows a **curve** and a straight line leaves large errors.

## 1. Why not use a straight line?

Simple Linear Regression tries to fit one straight line. It works well when the dots roughly follow a straight path.

But some patterns bend. For example, a car's fuel use may fall at first as speed rises, then rise again at high speed. A straight line misses this U-shaped pattern. Polynomial Regression can bend its prediction curve to match it better.

## 2. Polynomial degree

For one input `x`:

| Degree | Equation shape | Meaning |
|---:|---|---|
| 0 | `y_hat = b0` | A flat, constant prediction. |
| 1 | `y_hat = b0 + b1*x` | A straight line: ordinary linear regression. |
| 2 | `y_hat = b0 + b1*x + b2*x^2` | One smooth bend, such as a U-shape. |
| 3 or higher | Adds `x^3`, `x^4`, ... | More flexible curves. |

The **degree** controls how bendy the curve can be.

## 3. Why is it still called regression?

Polynomial Regression first makes new input columns. If the original input is `x`, degree 2 creates:

`x` and `x^2`

Then a normal Linear Regression model learns a weight for each column. So the curve is **non-linear in x**, but the model is still **linear in its coefficients** (`b0`, `b1`, `b2`).

With two inputs, a polynomial transformer can also create useful interaction features such as `x1*x2`.

## 4. Choosing the right degree

- Degree too low: the curve is too simple and **underfits**.
- Good degree: follows the real pattern and works on new data.
- Degree too high: bends around random noise and **overfits**.

Choose the degree using validation data or cross-validation. Do not choose it only because it looks perfect on the training dots.

## Visual diagram: line vs useful curve vs overfit curve

Run the next cell. The same U-shaped training data is given to three models. Degree 1 is too straight, degree 2 learns the main U-shape, and degree 9 bends too much to chase small random changes in the dots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Toy data with a U-shaped pattern
x_train = np.array([0.0, 0.8, 1.6, 2.4, 3.2, 4.1, 5.0, 5.8, 6.7, 8.0])
noise = np.array([0.8, -0.5, 0.3, -0.4, 0.2, -0.3, 0.4, -0.2, 0.5, -0.8])
y_train = 0.45 * (x_train - 4) ** 2 + 2 + noise
x_line = np.linspace(0, 8, 400).reshape(-1, 1)

models = [
    (1, 'Degree 1: underfit', 'Too straight', '#d62828'),
    (2, 'Degree 2: good fit', 'Captures the main U-shape', '#2a9d8f'),
    (9, 'Degree 9: overfit', 'Chases small noise changes', '#6a4c93')
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (degree, title, message, color) in zip(axes, models):
    polynomial = PolynomialFeatures(degree=degree, include_bias=False)
    x_train_polynomial = polynomial.fit_transform(x_train.reshape(-1, 1))
    x_line_polynomial = polynomial.transform(x_line)

    model = LinearRegression()
    model.fit(x_train_polynomial, y_train)
    y_line = model.predict(x_line_polynomial)

    ax.scatter(x_train, y_train, color='#1d3557', s=48, label='Training data', zorder=3)
    ax.plot(x_line, y_line, color=color, linewidth=2.6, label=f'Degree {degree} model')
    ax.set_title(title, color=color, weight='bold')
    ax.text(0.04, 0.05, message, transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.9))
    ax.set_xlabel('Input x')
    ax.grid(alpha=0.2)
axes[0].set_ylabel('Output y')
axes[0].legend(loc='upper right', fontsize=8)
plt.suptitle('Polynomial degree controls model flexibility', weight='bold', y=1.03)
plt.tight_layout()
plt.show()

# A quick look at the degree-2 input columns for x = 3
example = PolynomialFeatures(degree=2, include_bias=False).fit_transform([[3]])
print('For x = 3, degree-2 features are [x, x^2] =', example[0])


## 5. Simple Python recipe

1. Create polynomial features with `PolynomialFeatures(degree=2)`.
2. Fit `LinearRegression()` on the transformed features.
3. Transform any new input with the **same** polynomial transformer before predicting.
4. Compare degrees with validation or cross-validation, then keep the simplest degree that works well.

Do not use a huge degree by default. More flexibility can make training error look great while new-data error becomes worse.

## 6. Exam-ready recap

- Polynomial Regression is useful for curved, non-linear patterns.
- Degree 1 is ordinary linear regression.
- Degree 2 adds an `x^2` feature and can create one bend.
- Higher degrees are more flexible but can overfit.
- Select the degree using unseen validation data, not training fit alone.

**One-line answer:** Polynomial Regression adds powers of the input, such as `x^2`, so a linear regression model can fit a curved relationship.